### Import

In [1]:
import numpy as np
import pandas as pd
import datetime as dt

pd.options.display.max_rows = 1000
pd.options.display.max_colwidth = 10000
pd.set_option('display.max_columns', 500)

### general parameters 

In [2]:
mimiciv  = "PATH TO DATA/mimiciv(2.2)/"

### hosp - patients

In [3]:
patients = pd.read_csv(mimiciv + "hosp/patients.csv")
patients['dod'] = pd.to_datetime(patients['dod'])

In [4]:
patients.head(3)

In [5]:
print(patients.subject_id.nunique())

299712


### hosp - admission

In [6]:
admissions = pd.read_csv(mimiciv + "hosp/admissions.csv")

In [7]:
admissions.drop(columns=['admission_type', 'admit_provider_id', 'admission_location', 'discharge_location',
                'insurance', 'language', 'marital_status', 'edregtime', 'edouttime'], inplace=True)

In [8]:
admissions.head(3)

In [9]:
admissions['admittime'] = pd.to_datetime(admissions['admittime'])
admissions['dischtime'] = pd.to_datetime(admissions['dischtime'])
admissions['deathtime'] = pd.to_datetime(admissions['deathtime'])

In [10]:
admissions.head(3)

In [11]:
print(admissions.subject_id.nunique())
print(admissions.hadm_id.nunique())

180733
431231


### merge admission and patients

In [12]:
cohort_df = pd.merge(admissions, patients, on=['subject_id'], how='left')

In [13]:
cohort_df['death_time_disch'] = cohort_df['dod'] - cohort_df['dischtime']
cohort_df['death_time_disch'] = pd.to_timedelta(cohort_df['death_time_disch'])
cohort_df['death_time_disch'] = (cohort_df.death_time_disch / pd.Timedelta(days = 1)).round(0)
cohort_df['death_time_disch'] = cohort_df['death_time_disch'] + 1

In [14]:
cohort_df['age'] = (cohort_df['admittime'].dt.year - cohort_df['anchor_year']) + cohort_df['anchor_age']
cohort_df.drop(columns=['dod', 'anchor_year', 'anchor_age'], inplace=True)
cohort_df = cohort_df[['subject_id', 'hadm_id', 'gender', 'age', 'race', 'admittime', 'dischtime', 
                       'deathtime', 'hospital_expire_flag', 'death_time_disch', 'anchor_year_group']]

In [15]:
cohort_df.head(3)

In [16]:
print(cohort_df.subject_id.nunique())
print(cohort_df.hadm_id.nunique())

180733
431231


### icu - icustays

In [17]:
icustays = pd.read_csv(mimiciv + "icu/icustays.csv")

In [18]:
icustays.drop(columns=['first_careunit', 'last_careunit'], inplace=True)

In [19]:
icustays['intime']  = pd.to_datetime(icustays['intime'])
icustays['outtime'] = pd.to_datetime(icustays['outtime'])
icustays['los'] = icustays['los'].round(1)

In [20]:
icustays.head(3)

In [21]:
print(icustays.subject_id.nunique())
print(icustays.hadm_id.nunique())
print(icustays.stay_id.nunique())

50920
66239
73181


### merge cohort and icu

In [22]:
cohort_df = pd.merge(cohort_df, icustays, on=['subject_id', 'hadm_id'], how='left')

In [23]:
cohort_df['ADM_IN_DIFF'] = cohort_df['intime'] - cohort_df['admittime']
cohort_df['ADM_IN_DIFF'] = pd.to_timedelta(cohort_df['ADM_IN_DIFF'])
cohort_df['ADM_IN_DIFF'] = (cohort_df.ADM_IN_DIFF / pd.Timedelta(hours = 1)).round(1)

cohort_df['OUT_DIS_DIFF'] = cohort_df['dischtime'] - cohort_df['outtime']
cohort_df['OUT_DIS_DIFF'] = pd.to_timedelta(cohort_df['OUT_DIS_DIFF'])
cohort_df['OUT_DIS_DIFF'] = (cohort_df.OUT_DIS_DIFF / pd.Timedelta(hours = 1)).round(1)

In [24]:
cohort_df = cohort_df[~(((cohort_df.OUT_DIS_DIFF < -24) & (cohort_df.hospital_expire_flag == 0)))]

condition_1 = (cohort_df.ADM_IN_DIFF < 0)
cohort_df.loc[condition_1, 'admittime']  = cohort_df.loc[condition_1, 'intime']

condition_2 = ((cohort_df.OUT_DIS_DIFF < 0) & (cohort_df.hospital_expire_flag == 0))
cohort_df.loc[condition_2, 'dischtime']  = cohort_df.loc[condition_2, 'outtime']

cohort_df = cohort_df[cohort_df.OUT_DIS_DIFF > -48]

condition_3 = (cohort_df.OUT_DIS_DIFF < 0)
cohort_df.loc[condition_3, 'outtime']  = cohort_df.loc[condition_3, 'dischtime']

In [25]:
cohort_df.drop(columns=['OUT_DIS_DIFF', 'ADM_IN_DIFF'], inplace=True)

In [26]:
cohort_df = cohort_df[(cohort_df.stay_id.notnull()) & (cohort_df.intime >= cohort_df.admittime) & 
                      (cohort_df.outtime <= cohort_df.dischtime)]

In [27]:
cohort_df['icu_expire_flag'] = 0
cohort_df.loc[((cohort_df['deathtime'] > cohort_df['intime']) & (cohort_df['deathtime'] < cohort_df['outtime'] 
                                                              + pd.Timedelta(hours=6))), 'icu_expire_flag'] = 1

In [28]:
cohort_df['icuLos_h'] = cohort_df['outtime'] - cohort_df['intime']
cohort_df['icuLos_h'] = pd.to_timedelta(cohort_df['icuLos_h'])

cohort_df['icuLos_h'] = (cohort_df.icuLos_h / pd.Timedelta(hours = 1)).round(1)
cohort_df.rename(columns={"los": "icuLos_d"}, inplace=True)

In [29]:
cohort_df['hosp_Los'] = cohort_df['dischtime'] - cohort_df['admittime']
cohort_df['hosp_Los'] = pd.to_timedelta(cohort_df['hosp_Los'])

cohort_df['hospLos_d'] =  (cohort_df.hosp_Los / pd.Timedelta(days = 1)).round(1)
cohort_df['hospLos_h'] =  (cohort_df.hosp_Los / pd.Timedelta(hours= 1)).round(1)

cohort_df.drop(columns=['hosp_Los'], inplace=True)

In [30]:
cohort_df = cohort_df[['subject_id', 'hadm_id', 'stay_id', 'gender', 'age', 'race', 'intime', 'outtime',
                       'admittime', 'dischtime', 'icuLos_d', 'icuLos_h', 'hospLos_h', 'hospLos_d',
                       'icu_expire_flag', 'hospital_expire_flag', 'deathtime', 'death_time_disch',
                       'anchor_year_group']]

In [31]:
cohort_df.head(3)

In [32]:
print(cohort_df.subject_id.nunique())
print(cohort_df.hadm_id.nunique())
print(cohort_df.stay_id.nunique())
print(cohort_df[cohort_df.death_time_disch.notnull()].stay_id.nunique())

49988
64626
71432
27534


### Split for Extraction

In [33]:
def split_list(original_list, num_splits):
    
    split_size = len(original_list) // num_splits
    split_lists = [[] for _ in range(num_splits)]
    
    for i, item in enumerate(original_list):
        index = i // split_size
        if index >= num_splits:
            index = num_splits - 1
        split_lists[index].append(item)

    return split_lists

In [34]:
original_list = list(cohort_df.subject_id.unique())
split_lists = split_list(original_list, 3)

In [35]:
for i, lst in enumerate(split_lists):
    print(f"List {i+1} size: {len(lst)}")

List 1 size: 16662
List 2 size: 16662
List 3 size: 16664


In [36]:
chr1 = split_lists[0]
chr2 = split_lists[1]
chr3 = split_lists[2]

In [37]:
cohort1 = cohort_df[cohort_df.subject_id.isin(chr1)].copy()
cohort2 = cohort_df[cohort_df.subject_id.isin(chr2)].copy()
cohort3 = cohort_df[cohort_df.subject_id.isin(chr3)].copy()

In [38]:
print(cohort1.stay_id.nunique())
print(cohort2.stay_id.nunique())
print(cohort3.stay_id.nunique())

23791
23875
23766


In [39]:
cohort1_subject_id = list(cohort1.subject_id.unique())
cohort1_hadm_id = list(cohort1.hadm_id.unique())
cohort1_stay_id = list(cohort1.stay_id.unique())

cohort2_subject_id = list(cohort2.subject_id.unique())
cohort2_hadm_id = list(cohort2.hadm_id.unique())
cohort2_stay_id = list(cohort2.stay_id.unique())

cohort3_subject_id = list(cohort3.subject_id.unique())
cohort3_hadm_id = list(cohort3.hadm_id.unique())
cohort3_stay_id = list(cohort3.stay_id.unique())

In [40]:
cohort1_stay_id = [int(id) for id in cohort1_stay_id]
cohort2_stay_id = [int(id) for id in cohort2_stay_id]
cohort3_stay_id = [int(id) for id in cohort3_stay_id]

### Save Cohorts

In [41]:
with open("../Data/Cohort/cohort1_subject_id.txt", "w") as f:
    for subject_id in cohort1_subject_id:
        f.write(str(subject_id) +"\n")
        
with open("../Data/Cohort/cohort1_hadm_id.txt", "w") as f:
    for hadm_id in cohort1_hadm_id:
        f.write(str(hadm_id) +"\n")
        
with open("../Data/Cohort/cohort1_stay_id.txt", "w") as f:
    for stay_id in cohort1_stay_id:
        f.write(str(stay_id) +"\n")

In [42]:
with open("../Data/Cohort/cohort2_subject_id.txt", "w") as f:
    for subject_id in cohort2_subject_id:
        f.write(str(subject_id) +"\n")
        
with open("../Data/Cohort/cohort2_hadm_id.txt", "w") as f:
    for hadm_id in cohort2_hadm_id:
        f.write(str(hadm_id) +"\n")
        
with open("../Data/Cohort/cohort2_stay_id.txt", "w") as f:
    for stay_id in cohort2_stay_id:
        f.write(str(stay_id) +"\n")

In [43]:
with open("../Data/Cohort/cohort3_subject_id.txt", "w") as f:
    for subject_id in cohort3_subject_id:
        f.write(str(subject_id) +"\n")
        
with open("../Data/Cohort/cohort3_hadm_id.txt", "w") as f:
    for hadm_id in cohort3_hadm_id:
        f.write(str(hadm_id) +"\n")
        
with open("../Data/Cohort/cohort3_stay_id.txt", "w") as f:
    for stay_id in cohort3_stay_id:
        f.write(str(stay_id) +"\n")

### Selective Cohort Selection

### radiology reports

In [29]:
radiology_note = pd.read_csv(mimiciv + "note/radiology.csv")

In [30]:
radiology_note = radiology_note[['note_id', 'subject_id', 'charttime']]
cohort_copy = cohort_df.copy()
cohort_copy = cohort_copy[['subject_id', 'hadm_id', 'stay_id', 'intime', 'outtime']].reset_index(drop=True)
radiology_note_in_icu = pd.merge(radiology_note, cohort_copy, on=['subject_id'], how='left')
radiology_note_in_icu = radiology_note_in_icu[((radiology_note_in_icu.charttime >= radiology_note_in_icu.intime) &
                                               (radiology_note_in_icu.charttime <= radiology_note_in_icu.outtime))]

In [31]:
radiology_note_in_icu = radiology_note_in_icu[['subject_id', 'hadm_id', 'stay_id', 'note_id', 
                                               'charttime', 'intime']]

radiology_note_in_icu['charttime'] = pd.to_datetime(radiology_note_in_icu['charttime'])
radiology_note_in_icu['intime'] = pd.to_datetime(radiology_note_in_icu['intime'])

radiology_note_in_icu['note_since_admitt'] = radiology_note_in_icu['charttime'] - radiology_note_in_icu['intime']
radiology_note_in_icu['note_since_admitt'] = pd.to_timedelta(radiology_note_in_icu['note_since_admitt'])
radiology_note_in_icu['note_since_admitt'] = (radiology_note_in_icu.note_since_admitt / pd.Timedelta(hours=1)).round(1)
radiology_note_in_icu.drop(columns=['intime'], inplace=True)

radiology_note_in_icu.reset_index(inplace=True, drop=True)

In [32]:
radiology_note_in_icu.head(3)

In [33]:
print(radiology_note_in_icu.subject_id.nunique())
print(radiology_note_in_icu.hadm_id.nunique())
print(radiology_note_in_icu.stay_id.nunique())
print(radiology_note_in_icu.note_id.nunique())

36008
43509
47365
205455


### CXR

In [34]:
cxrmetadata = pd.read_csv(mimiciv + "cxr/mimic-cxr-2.0.0-metadata.csv")

In [35]:
cxrmetadata['StudyDate'] = pd.to_datetime(cxrmetadata['StudyDate'], format='%Y%m%d')
cxrmetadata['StudyTime'] = cxrmetadata.apply(lambda x : '%#010.3f' % x['StudyTime'] ,1)
cxrmetadata['StudyTime'] = pd.to_datetime(cxrmetadata['StudyTime'], format='%H%M%S.%f').dt.time
cxrmetadata['cxrtime']   = cxrmetadata.apply(lambda r : dt.datetime.combine(r['StudyDate'],r['StudyTime']),1)
cxrmetadata['cxrtime']   = cxrmetadata['cxrtime'].dt.floor('Min')

In [36]:
cxrmetadata = cxrmetadata[['subject_id', 'study_id', 'dicom_id', 'cxrtime']]
cxr_in_icu = pd.merge(cxrmetadata, cohort_copy, on=['subject_id'], how='left')
cxr_in_icu = cxr_in_icu[((cxr_in_icu.cxrtime >= cxr_in_icu.intime) &
                         (cxr_in_icu.cxrtime <= cxr_in_icu.outtime))]
cxr_in_icu = cxr_in_icu[['subject_id', 'hadm_id', 'stay_id', 'study_id', 'dicom_id', 'cxrtime', 'intime']]

cxr_in_icu['cxrtime'] = pd.to_datetime(cxr_in_icu['cxrtime'])
cxr_in_icu['intime'] = pd.to_datetime(cxr_in_icu['intime'])

cxr_in_icu['cxr_since_admitt'] = cxr_in_icu['cxrtime'] - cxr_in_icu['intime']
cxr_in_icu['cxr_since_admitt'] = pd.to_timedelta(cxr_in_icu['cxr_since_admitt'])
cxr_in_icu['cxr_since_admitt'] = (cxr_in_icu.cxr_since_admitt / pd.Timedelta(hours=1)).round(1)
cxr_in_icu.drop(columns=['intime'], inplace=True)

cxr_in_icu.reset_index(inplace=True, drop=True)

In [37]:
cxr_in_icu.head(2)

In [38]:
print(cxr_in_icu.subject_id.nunique())
print(cxr_in_icu.hadm_id.nunique())
print(cxr_in_icu.stay_id.nunique())
print(cxr_in_icu.study_id.nunique())
print(cxr_in_icu.dicom_id.nunique())

9492
11117
12067
38034
43082


### Filter based on recording time of Radiology Reports since icu admission

In [39]:
first_x_hour = 48

radiology_note_in_icu = radiology_note_in_icu[radiology_note_in_icu.note_since_admitt <= first_x_hour]

In [40]:
print(radiology_note_in_icu.subject_id.nunique())
print(radiology_note_in_icu.hadm_id.nunique())
print(radiology_note_in_icu.stay_id.nunique())
print(radiology_note_in_icu.note_id.nunique())

35695
43008
46733
118540


### Filter based on recording time of CXR since icu admission

In [41]:
first_x_hour = 48

cxr_in_icu = cxr_in_icu[cxr_in_icu.cxr_since_admitt <= first_x_hour]

In [42]:
print(cxr_in_icu.subject_id.nunique())
print(cxr_in_icu.hadm_id.nunique())
print(cxr_in_icu.stay_id.nunique())
print(cxr_in_icu.study_id.nunique())
print(cxr_in_icu.dicom_id.nunique())

9062
10558
11415
19566
22043


### Filter based on CXR and Radiology Reports

In [43]:
stay_id_report_list = list(radiology_note_in_icu.stay_id.unique())
stay_id_cxr_list = list(cxr_in_icu.stay_id.unique())
report_cxr_intersect = list(set(stay_id_report_list).intersection(stay_id_cxr_list))
report_cxr_union = list(set(stay_id_report_list + stay_id_cxr_list))

In [44]:
cxr_report_filter = 'union' # 'report' - 'cxr' - 'both' - 'union'

In [45]:
if cxr_report_filter == 'report':
    cohort_df = cohort_df[cohort_df.stay_id.isin(stay_id_report_list)]
elif cxr_report_filter == 'cxr':
    cohort_df = cohort_df[cohort_df.stay_id.isin(stay_id_cxr_list)]
elif cxr_report_filter == 'both':
    cohort_df = cohort_df[cohort_df.stay_id.isin(report_cxr_intersect)]
elif cxr_report_filter == 'union':
    cohort_df = cohort_df[cohort_df.stay_id.isin(report_cxr_union)]

In [46]:
print(cohort_df.subject_id.nunique())
print(cohort_df.hadm_id.nunique())
print(cohort_df.stay_id.nunique())

35704
43025
46756


### Filter based on Age

In [47]:
min_age = 18
max_age = 100

cohort_df = cohort_df[(cohort_df.age >= min_age) & (cohort_df.age <= max_age)]

In [48]:
print(cohort_df.subject_id.nunique())
print(cohort_df.hadm_id.nunique())
print(cohort_df.stay_id.nunique())

35704
43025
46756


### Filter based on ICU length of stay

In [49]:
min_los = 48

cohort_df = cohort_df[cohort_df.icuLos_h >= min_los]

In [50]:
print(cohort_df.subject_id.nunique())
print(cohort_df.hadm_id.nunique())
print(cohort_df.stay_id.nunique())

20908
24009
25736


### Filter based on first Hospital and ICU Admission

In [51]:
cohort_df.sort_values(by=['subject_id', 'admittime', 'intime'], inplace=True)

In [52]:
cohort_df = cohort_df.groupby(['subject_id']).head(1)
cohort_df['stay_id'] = cohort_df['stay_id'].astype(int)

In [54]:
print(cohort_df.subject_id.nunique())
print(cohort_df.hadm_id.nunique())
print(cohort_df.stay_id.nunique())

20908
20908
20908


### Final Admission ID

In [55]:
final_subject_id = list(cohort_df.subject_id.unique())
final_hadm_id = list(cohort_df.hadm_id.unique())
final_stay_id = list(cohort_df.stay_id.unique())

### Write patients admission ids

In [56]:
# with open("../Data/Cohort/subject_id_cohort.txt", "w") as f:
#     for subject_id in final_subject_id:
#         f.write(str(subject_id) +"\n")

In [57]:
# with open("../Data/Cohort/hadm_id_cohort.txt", "w") as f:
#     for hadm_id in final_hadm_id:
#         f.write(str(hadm_id) +"\n")

In [58]:
# with open("../Data/Cohort/stay_id_cohort.txt", "w") as f:
#     for stay_id in final_stay_id:
#         f.write(str(stay_id) +"\n")